### Test Envirnment 

- Step 1 → API connection
- Step 2 → PDF text extraction
- Step 3 → Text chunking
- Step 4 → Embeddings API
- Step 5 → FAISS vector index
- Step 6 → Retrieval
- Step 7 → RAG prompt
- Step 8 → LLM answer

#### Step 1 → API connection

In [1]:
from dotenv import load_dotenv
import os
from huggingface_hub import InferenceClient

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN not found in .env")


client = InferenceClient(api_key=HF_TOKEN)


In [2]:
# messages = [
#     {   "role" : "user",
#         "content" : "explain what is RAG in one line ?"
#     }
# ]

# response = client.chat_completion(
#     model="MiniMaxAI/MiniMax-M2.5",
#     messages=messages,
#     max_tokens=50
# )

# print(response)

In [3]:
# import json 
# print(json.dumps(response, indent=4))


In [4]:
# print(response.choices[0].message.content)

#### Step 2 → PDF text extraction

In [5]:
import fitz #PyMuPdf
import os 

pdf_path = "D:/Code/Python/Intership_Learning/AI_Ml_Projects/RAG_Q&A_Bot/PDFs/Introduction to Outlier.docx.pdf"

doc = fitz.open(pdf_path) #open in PyMuPdf

pdf_text = ""

for page in doc:
    pdf_text+= page.get_text() + "\n"

page_no = doc.page_count
doc.close()

print(f"PDF Loaded Successfully. \nDocument Total Pages : {page_no}\nNumber of character extracted : {len(pdf_text)}\n")


PDF Loaded Successfully. 
Document Total Pages : 22
Number of character extracted : 18680



#### Note :
- Sometimes PyMuPDF DLL causes import errors on Windows 11. 

- To fix: go to Defender → Virus & Threat Protection → Ransomware Protection → Manage Ransomware Protection → Allow an app through Controlled Folder Access → add your python.exe file.

- Restart VS code editor or Kernal

#### Step 3 → Text chunking

In [6]:
# Logic of chunk :
def chunk_text(text, size = 500, overlap=50):
    """
    Splits text into overlapping chunks.
    text: full string
    size: characters per chunk
    overlap: characters to overlap between chunks
    """

    chunks = []
    start = 0

    while start < len(text):
        end = start + size
        chunk = text[start:end]
        chunks.append(chunk)

        start+= size - overlap #for continuity

    return chunks

# Create Chunks:
chunks = chunk_text(pdf_text, size = 500, overlap = 50)

print(f"PDF split into {len(chunks)} chunks.")
print(f"Example chunk :\n{chunks[0]}")


PDF split into 42 chunks.
Example chunk :
Introduction to Outlier 
An outlier is an observation that is unlike the other 
observations. It is rare, or distinct, or does not fit in some way. 
It is also called anomalies. 
 
Outliers are extreme values or unusual data points in a dataset 
that differ significantly from other observations. They are 
crucial to understand because they can affect model accuracy 
and lead to misleading insights if not properly addressed.  
Types of Outliers 
●​ Univariate Outliers: These are unusual values in


#### Step 4 → Embeddings API

In [7]:
import torch

print(torch.cuda.device_count())  # Output: 1
for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_name(i))  # Output: NVIDIA GeForce RTX 3050

1
NVIDIA GeForce RTX 3050 Laptop GPU


In [8]:
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

# Load model on GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

def get_embedding(text):
    # Returns embedding on CPU as numpy array
    embedding = embedding_model.encode(text, convert_to_numpy=True)
    return embedding

# Example usage
chunks = ["This is a test.", "Another chunk."]
embeddings = [get_embedding(chunk) for chunk in chunks]

embeddings = np.vstack(embeddings)
print(embeddings.shape)

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

d:\Code\AI_ML_env\ai_env\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(2, 384)


#### Step 5 → FAISS vector index